# UIESS – Underwater Image Enhancement (Kaggle)
Run all cells top-to-bottom. GPU accelerator recommended.

**Dataset path assumed:** `/kaggle/input/uieb-dataset/UIESS-master/`

In [ ]:
# ── CELL 1 · Install dependencies ─────────────────────────────────────────
!pip install -q torchvision seaborn scikit-learn

In [ ]:
# ── CELL 2 · Copy project to writable directory ───────────────────────────
import os, shutil

SRC  = '/kaggle/input/uieb-dataset/UIESS-master'
DEST = '/kaggle/working/UIESS-master'

if os.path.exists(DEST):
    shutil.rmtree(DEST)
shutil.copytree(SRC, DEST)

os.chdir(DEST)
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# ── CELL 3 · Verify GPU and imports ───────────────────────────────────────
import sys
sys.path.insert(0, DEST)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Verify all project modules are importable
import models, datasets, loss
print('Project modules imported successfully.')

In [ ]:
# ── CELL 4 · Run inference (test_REAL_image) ──────────────────────────────
# Paths are already baked into test.py defaults.
# Pass --test_dir / --model_dir / --out_dir explicitly only if you changed
# the dataset name on Kaggle.

!python test.py \
    --exp_name   release \
    --test_dir   /kaggle/input/uieb-dataset/UIESS-master/data/testA \
    --model_dir  /kaggle/input/uieb-dataset/UIESS-master/saved_models/release \
    --out_dir    /kaggle/working/output \
    --checkpoint 29

In [ ]:
# ── CELL 5 · Display output images ────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

OUT_DIR = Path('/kaggle/working/output/release/test_REAL_image/29')

image_paths = sorted(OUT_DIR.glob('*.*'))[:8]   # show up to 8 images

if not image_paths:
    print('No output images found in', OUT_DIR)
else:
    n     = len(image_paths)
    ncols = min(4, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
    axes = [axes] if n == 1 else axes.flatten()

    for ax, img_path in zip(axes, image_paths):
        img = mpimg.imread(str(img_path))
        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=8)
        ax.axis('off')

    # Hide unused axes
    for ax in axes[n:]:
        ax.axis('off')

    plt.suptitle('Enhanced Underwater Images', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()
    print(f'Displayed {n} image(s) from {OUT_DIR}')

In [ ]:
# ── CELL 6 (OPTIONAL) · Side-by-side: input vs enhanced ──────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

INPUT_DIR = Path('/kaggle/input/uieb-dataset/UIESS-master/data/testA')
ENAHCED_DIR = Path('/kaggle/working/output/release/test_REAL_image/29')

pairs = []
for inp in sorted(INPUT_DIR.glob('*.*'))[:4]:   # first 4 pairs
    out = ENAHCED_DIR / inp.name
    if out.exists():
        pairs.append((inp, out))

if pairs:
    fig, axes = plt.subplots(len(pairs), 2, figsize=(10, 5 * len(pairs)))
    if len(pairs) == 1:
        axes = [axes]
    for row, (inp, out) in zip(axes, pairs):
        row[0].imshow(mpimg.imread(str(inp)));  row[0].set_title('Input');    row[0].axis('off')
        row[1].imshow(mpimg.imread(str(out)));  row[1].set_title('Enhanced'); row[1].axis('off')
    plt.suptitle('Input vs Enhanced', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No matching input/output pairs found.')

In [ ]:
# ── CELL 7 (OPTIONAL) · Run training ─────────────────────────────────────
# Remove the leading '#' to execute.
# Training writes checkpoints to /kaggle/working/output/saved_models/

# !python train.py \
#     --data_root  /kaggle/input/uieb-dataset/UIESS-master/data \
#     --out_dir    /kaggle/working/output \
#     --exp_name   my_experiment \
#     --n_epochs   35 \
#     --batch_size 4 \
#     --lr         5e-4